# 02 — Базовая модель

In [1]:
from __future__ import annotations

import gc
import json
import math
import hashlib
import shutil
import subprocess
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 42
np.random.seed(SEED)

HORIZON = 30
EXPECTED_USERS = 250_000
ANCHOR_STRIDE_DAYS = 42

WINDOWS = [7, 14, 30, 60, 90, 180, 270, 365]
STATS_WINDOWS = [30, 90, 180, 365]
WEEK_WINDOWS = [90, 180, 365]
SPAN_WINDOWS = [180, 365]
CLEAN_WINDOWS = [30, 60, 90, 180, 270, 365]
NEW_YEAR_WINDOWS = [30, 60, 90, 180, 270, 365]

V9_CANONICAL_POOL = [
    date(2025, 2, 13),
    date(2025, 3, 26),
    date(2025, 5, 7),
    date(2025, 6, 18),
    date(2025, 7, 30),
    date(2025, 9, 10),
    date(2025, 10, 22),
    date(2025, 11, 12),
    date(2025, 12, 3),
    date(2025, 12, 15),
    date(2026, 1, 14),
]

PRIMARY_OUTERS = [
    date(2025, 7, 30),
    date(2025, 9, 10),
    date(2025, 10, 22),
    date(2026, 1, 14),
]
STRESS_OUTERS = [
    date(2025, 6, 18),
    date(2025, 12, 3),
    date(2025, 12, 15),
]

RECENT_MODEL_N_ANCHORS = 4
RECENCY_DECAY_ATTR = 180.0
RECENCY_DECAY_DAYS = RECENCY_DECAY_ATTR

CAPACITY = {"recent": 1138, "classifier": 820, "positive": 845}
FIXED_RECENT_WEIGHT = 0.46619
FIXED_HURDLE_WEIGHT = 0.53381
SEED_RECENT = 143
SEED_CLASSIFIER = 243
SEED_POSITIVE = 343
CONTROL_GAMMA = 2.0

ARCH_K = 5
QUALITY_COL = "future_arch_quality_score"
GATE_COLS = [f"future_arch5_w_{j}" for j in range(ARCH_K)]

PURCHASE_AUX_ITERATIONS = 500
PURCHASE_AUX_DEPTH = 6
PURCHASE_AUX_LR = 0.04
PURCHASE_AUX_DECAY_DAYS = 180.0
PURCHASE_AUX_SEED = 1343

LOW_VALUE_BLOCK_UNTIL_RANK = 0.60
LOW_VALUE_FULL_V12_FROM_RANK = 0.90
CANDIDATE = "B3_PURCHASE_AUX_LOW"

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
FEATURE_DIR = PROJECT_DIR / "artifacts" / "prepare_features"
ARTIFACT_ROOT = PROJECT_DIR / "artifacts" / "base_model"
MODEL_FRAME_CACHE_DIR = ARTIFACT_ROOT / "_cache" / "model_frames"
V12_TARGET_CACHE_DIR = ARTIFACT_ROOT / "_cache" / "targets"
PURCHASE_AUX_CACHE_DIR = ARTIFACT_ROOT / "_cache" / "purchase_aux"

for p in [ARTIFACT_ROOT, MODEL_FRAME_CACHE_DIR, V12_TARGET_CACHE_DIR, PURCHASE_AUX_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CACHE_MODEL_FRAMES = True
CACHE_DIR = ARTIFACT_ROOT / "_cache" / "v9"
CACHE_EXTRA_DIR = ARTIFACT_ROOT / "_cache" / "cd_extra"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_EXTRA_DIR.mkdir(parents=True, exist_ok=True)
FORCE_REBUILD_EXTRA = False

def has_nvidia_gpu() -> bool:
    try:
        return (
            shutil.which("nvidia-smi") is not None
            and subprocess.run(
                ["nvidia-smi", "-L"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                timeout=5,
                check=False,
            ).returncode == 0
        )
    except Exception:
        return False

def gpu_total_memory_mb() -> int:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
        text=True,
        timeout=5,
    ).strip().splitlines()
    return int(out[0])

if not has_nvidia_gpu():
    raise RuntimeError("NVIDIA GPU/CUDA not detected")

TASK_TYPE = "GPU"
DEVICES = "0"
GPU_MEMORY_MB = gpu_total_memory_mb()
BORDER_COUNT = 128 if GPU_MEMORY_MB < 11_000 else 254

smoke = CatBoostRegressor(
    iterations=2,
    depth=2,
    learning_rate=0.1,
    loss_function="RMSE",
    task_type="GPU",
    devices=DEVICES,
    verbose=False,
    allow_writing_files=False,
    random_seed=SEED,
)
smoke.fit(
    np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32),
    np.array([0.0, 1.0, 2.0, 3.0], dtype=np.float32),
)
del smoke
gc.collect()

print("GPU:", subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0])
print("GPU memory MB:", GPU_MEMORY_MB)


GPU: NVIDIA GeForce RTX 4070 SUPER
GPU memory MB: 12282


In [2]:
def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("\n".join(str(p) for p in paths))

TRAIN_PATH = first_existing(
    DATA_DIR / "train.parquet",
    PROJECT_DIR / "train.parquet",
)
SAMPLE_PATH = first_existing(
    DATA_DIR / "sample_submit.csv",
    DATA_DIR / "sample_submission.csv",
    PROJECT_DIR / "sample_submit.csv",
    PROJECT_DIR / "sample_submission.csv",
)
TEMP_HIST_PATH = first_existing(
    FEATURE_DIR / "temporal_features_historical_K5.parquet",
)
TEMP_FINAL_PATH = first_existing(
    FEATURE_DIR / "temporal_features_final_K5_2026-02-13.parquet",
)

data = pl.read_parquet(TRAIN_PATH)
if data.schema["event_date"] != pl.Date:
    data = data.with_columns(pl.col("event_date").cast(pl.Date))

required_train = {
    "event_date", "user_id", "search", "cat",
    "has_search_to_cart", "has_search_to_ord",
    "has_cat_to_cart", "has_cat_to_ord",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_search", "gmv_cat",
    "to_cart", "to_ord", "gmv", "searches",
}
missing = sorted(required_train - set(data.columns))
if missing:
    raise RuntimeError(f"Missing train columns: {missing}")

min_date = data["event_date"].min()
max_date = data["event_date"].max()
submit_template = pl.read_csv(SAMPLE_PATH)

assert submit_template.columns == ["user_id", "predict"]
assert submit_template.height == EXPECTED_USERS
assert submit_template["user_id"].n_unique() == EXPECTED_USERS
assert max_date == date(2026, 2, 13)

all_users = submit_template.select("user_id")

print("train:", data.shape)
print("period:", min_date, "->", max_date)
print("users:", all_users.height)

HOLIDAY_CODES = {
    221, 222, 223,
    306, 307, 308,
    429, 430, 501,
    507, 508, 509,
    610, 611, 612,
    1102, 1103, 1104,
}


train: (30631006, 18)
period: 2025-01-01 -> 2026-02-13
users: 250000


## Признаки

In [3]:
def monthday_code(d: date) -> int:
    return d.month * 100 + d.day

def shift_year(d: date, delta_years: int = -1) -> date:
    target_year = d.year + delta_years
    try:
        return d.replace(year=target_year)
    except ValueError:
        return d.replace(year=target_year, day=28)

def is_newyear_ramp_day(d: date) -> bool:
    return d.month == 12 and d.day >= 15

def is_newyear_post_day(d: date) -> bool:
    return d.month == 1 and d.day <= 3

def is_newyear_window_day(d: date) -> bool:
    return is_newyear_ramp_day(d) or is_newyear_post_day(d)

def count_newyear_days(start: date, end: date) -> int:
    if end < start:
        return 0
    return sum(
        is_newyear_window_day(start + timedelta(days=i))
        for i in range((end - start).days + 1)
    )

def count_pattern_days(start: date, end: date) -> int:
    if end < start:
        return 0
    return sum(
        monthday_code(start + timedelta(days=i)) in HOLIDAY_CODES
        for i in range((end - start).days + 1)
    )

def count_special_days(start: date, end: date) -> int:
    if end < start:
        return 0
    return sum(
        (
            monthday_code(start + timedelta(days=i)) in HOLIDAY_CODES
            or is_newyear_window_day(start + timedelta(days=i))
        )
        for i in range((end - start).days + 1)
    )

def days_since_last_newyear_window(anchor: date) -> int:
    current_end = date(anchor.year, 1, 3)
    if anchor >= current_end:
        return (anchor - current_end).days
    return (anchor - date(anchor.year - 1, 1, 3)).days

def future_calendar_values(anchor: date) -> dict[str, float]:
    days = [anchor + timedelta(days=i) for i in range(1, HORIZON + 1)]
    mid = days[len(days) // 2]
    angle = 2.0 * math.pi * mid.timetuple().tm_yday / 365.25

    return {
        'target_holiday_pattern_days': float(sum(monthday_code(d) in HOLIDAY_CODES for d in days)),
        'target_newyear_ramp_days': float(sum(is_newyear_ramp_day(d) for d in days)),
        'target_newyear_post_days': float(sum(is_newyear_post_day(d) for d in days)),
        'target_newyear_window_days': float(sum(is_newyear_window_day(d) for d in days)),
        'target_special_days': float(sum(monthday_code(d) in HOLIDAY_CODES or is_newyear_window_day(d) for d in days)),
        'target_weekend_days': float(sum(d.weekday() >= 5 for d in days)),
        'target_mid_doy_sin': float(math.sin(angle)),
        'target_mid_doy_cos': float(math.cos(angle)),
        'days_since_last_newyear_window': float(days_since_last_newyear_window(anchor)),
    }

month_expr = pl.col('event_date').dt.month().cast(pl.Int32)
day_expr = pl.col('event_date').dt.day().cast(pl.Int32)
date_code_expr = month_expr * 100 + day_expr

is_empirical_expr = date_code_expr.is_in(sorted(HOLIDAY_CODES))
is_newyear_ramp_expr = (month_expr == 12) & (day_expr >= 15)
is_newyear_post_expr = (month_expr == 1) & (day_expr <= 3)
is_newyear_window_expr = is_newyear_ramp_expr | is_newyear_post_expr

data = data.with_columns([
    is_empirical_expr.alias('is_empirical_holiday'),
    is_newyear_ramp_expr.alias('is_newyear_ramp'),
    is_newyear_post_expr.alias('is_newyear_post'),
    is_newyear_window_expr.alias('is_newyear_window'),
    (is_empirical_expr | is_newyear_window_expr).alias('is_special_day'),
])

DAILY_GLOBAL = (
    data.group_by('event_date')
    .agg([
        pl.col('gmv').sum().alias('global_gmv'),
        pl.col('to_ord').sum().alias('global_orders'),
        pl.col('to_cart').sum().alias('global_cart'),
        pl.col('searches').sum().alias('global_searches'),
        pl.col('user_id').n_unique().alias('global_active_users'),
    ])
    .sort('event_date')
)

def _global_sum(start: date, end: date, col: str) -> float:
    if end < start:
        return 0.0
    part = DAILY_GLOBAL.filter(pl.col('event_date').is_between(start, end, closed='both'))
    return float(part[col].sum()) if part.height else 0.0

def global_state_values(anchor: date) -> dict[str, float]:
    out: dict[str, float] = {}

    for w in [7, 14, 30, 60, 90, 180, 365]:
        start = max(min_date, anchor - timedelta(days=w - 1))
        days = max(1, (anchor - start).days + 1)
        part = DAILY_GLOBAL.filter(pl.col('event_date').is_between(start, anchor, closed='both'))

        gmv = float(part['global_gmv'].sum()) if part.height else 0.0
        orders = float(part['global_orders'].sum()) if part.height else 0.0
        searches = float(part['global_searches'].sum()) if part.height else 0.0
        active_mean = float(part['global_active_users'].mean()) if part.height else 0.0

        out[f'global_gmv_runrate_{w}'] = gmv * 30.0 / days
        out[f'global_orders_runrate_{w}'] = orders * 30.0 / days
        out[f'global_searches_runrate_{w}'] = searches * 30.0 / days
        out[f'global_active_users_mean_{w}'] = active_mean
        out[f'global_gmv_per_active_{w}'] = (
            gmv / max(float(part['global_active_users'].sum()), 1.0)
            if part.height else 0.0
        )

    def ratio_recent_vs_prev(w: int, col: str) -> float:
        recent_start = anchor - timedelta(days=w - 1)
        prev_end = recent_start - timedelta(days=1)
        prev_start = prev_end - timedelta(days=w - 1)
        recent = _global_sum(max(min_date, recent_start), anchor, col)
        prev = _global_sum(max(min_date, prev_start), prev_end, col)
        return float(np.clip((recent + 1.0) / (prev + 1.0), 0.25, 4.0))

    out['global_gmv_trend_7'] = ratio_recent_vs_prev(7, 'global_gmv')
    out['global_gmv_trend_30'] = ratio_recent_vs_prev(30, 'global_gmv')
    out['global_gmv_trend_90'] = ratio_recent_vs_prev(90, 'global_gmv')
    out['global_orders_trend_30'] = ratio_recent_vs_prev(30, 'global_orders')
    out['global_searches_trend_30'] = ratio_recent_vs_prev(30, 'global_searches')
    return out

FINAL_CALENDAR = future_calendar_values(max_date)
SEASONAL_MATCH_ANCHOR = shift_year(max_date, -1)
SEASONAL_CALENDAR = future_calendar_values(SEASONAL_MATCH_ANCHOR)

print('Final target:', max_date + timedelta(days=1), '→', max_date + timedelta(days=HORIZON))
print('Final calendar:', FINAL_CALENDAR)
print('Seasonal analogue anchor:', SEASONAL_MATCH_ANCHOR)
print('Seasonal analogue target:', SEASONAL_MATCH_ANCHOR + timedelta(days=1), '→', SEASONAL_MATCH_ANCHOR + timedelta(days=HORIZON))
print('Seasonal analogue calendar:', SEASONAL_CALENDAR)


Final target: 2026-02-14 → 2026-03-15
Final calendar: {'target_holiday_pattern_days': 6.0, 'target_newyear_ramp_days': 0.0, 'target_newyear_post_days': 0.0, 'target_newyear_window_days': 0.0, 'target_special_days': 6.0, 'target_weekend_days': 10.0, 'target_mid_doy_sin': 0.8584015232918917, 'target_mid_doy_cos': 0.512978386299228, 'days_since_last_newyear_window': 41.0}
Seasonal analogue anchor: 2025-02-13
Seasonal analogue target: 2025-02-14 → 2025-03-15
Seasonal analogue calendar: {'target_holiday_pattern_days': 6.0, 'target_newyear_ramp_days': 0.0, 'target_newyear_post_days': 0.0, 'target_newyear_window_days': 0.0, 'target_special_days': 6.0, 'target_weekend_days': 9.0, 'target_mid_doy_sin': 0.8584015232918917, 'target_mid_doy_cos': 0.512978386299228, 'days_since_last_newyear_window': 41.0}


In [4]:
def rmse_log(y_log, pred_log, weight=None) -> float:
    y_log = np.asarray(y_log, dtype=np.float64)
    pred_log = np.asarray(pred_log, dtype=np.float64)
    err2 = (y_log - pred_log) ** 2
    if weight is None:
        return float(np.sqrt(np.mean(err2)))
    weight = np.asarray(weight, dtype=np.float64)
    return float(np.sqrt(np.average(err2, weights=weight)))

RECENCY_SENTINEL = max(WINDOWS) + 31

def _date_diff_expr(anchor: date, condition: pl.Expr, name: str) -> pl.Expr:
    last_day = pl.col("event_date").filter(condition).max().cast(pl.Int32)
    return (pl.lit(anchor).cast(pl.Date).cast(pl.Int32) - last_day).alias(name)

def available_calendar_days(anchor: date, w: int) -> int:
    start = max(min_date, anchor - timedelta(days=w - 1))
    return max(1, (anchor - start).days + 1)

def recency_weights(frame: pl.DataFrame, decay_days: float = RECENCY_DECAY_ATTR) -> np.ndarray:
    anchor_days = frame["anchor_date"].cast(pl.Int32).to_numpy()
    newest = int(np.max(anchor_days))
    age = newest - anchor_days
    w = np.exp(-age / decay_days).astype(np.float32)
    return w / max(float(w.mean()), 1e-12)

def temporal_weights_against_anchor_pool(
    frame: pl.DataFrame,
    reference_anchors: list[date],
    decay_days: float = RECENCY_DECAY_ATTR,
) -> np.ndarray:
    newest = max(reference_anchors)
    newest_day = (newest - date(1970, 1, 1)).days
    anchor_days = frame["anchor_date"].cast(pl.Int32).to_numpy()
    raw = np.exp(-(newest_day - anchor_days) / decay_days).astype(np.float32)
    norm = float(np.mean([
        math.exp(-(newest - a).days / decay_days)
        for a in reference_anchors
    ]))
    return raw / max(norm, 1e-12)


In [5]:
SUM_COLS = [
    "search", "cat",
    "has_search_to_cart", "has_search_to_ord",
    "has_cat_to_cart", "has_cat_to_ord",
    "search_to_cart", "search_to_ord",
    "cat_to_cart", "cat_to_ord",
    "gmv_search", "gmv_cat",
    "to_cart", "to_ord", "gmv", "searches",
]

CORE_FEATURE_NAMES = [
    "search_to_cart_rate",
    "search_to_order_rate",
    "cat_to_cart_rate",
    "searches_per_active_day",
    "recency_days",
    "active_days",
    "orders_recent_share",
]

def make_features_for_anchor(
    data: pl.DataFrame,
    all_users: pl.DataFrame,
    anchor: date,
) -> pl.DataFrame:
    max_window = max(WINDOWS)
    hist_start = max(min_date, anchor - timedelta(days=max_window - 1))

    hist = data.filter(
        pl.col("event_date").is_between(hist_start, anchor, closed="both")
    )

    exprs: list[pl.Expr] = []

    for w in WINDOWS:
        start = anchor - timedelta(days=w - 1)
        mask = pl.col("event_date") >= pl.lit(start)

        exprs.extend([
            pl.col("event_date").filter(mask).count().alias(f"active_days_{w}"),
            pl.col("event_date")
              .filter(mask & (pl.col("to_ord") > 0))
              .count()
              .alias(f"order_days_{w}"),
            pl.col("gmv").filter(mask).max().alias(f"gmv_max_{w}"),
        ])

        if w in STATS_WINDOWS:
            exprs.extend([
                pl.col("gmv").filter(mask).mean().alias(f"gmv_mean_active_{w}"),
                pl.col("gmv").filter(mask).std().alias(f"gmv_std_active_{w}"),
                pl.col("to_ord").filter(mask).std().alias(f"orders_std_active_{w}"),
            ])

        if w in WEEK_WINDOWS:
            exprs.extend([
                pl.col("event_date")
                  .filter(mask)
                  .dt.truncate("1w")
                  .n_unique()
                  .alias(f"active_weeks_{w}"),
                pl.col("event_date")
                  .filter(mask & (pl.col("to_ord") > 0))
                  .dt.truncate("1w")
                  .n_unique()
                  .alias(f"order_weeks_{w}"),
            ])

        for c in SUM_COLS:
            exprs.append(
                pl.col(c).filter(mask).sum().alias(f"{c}_sum_{w}")
            )

        if w in CLEAN_WINDOWS:
            clean_mask = mask & (~pl.col("is_special_day"))
            exprs.extend([
                pl.col("gmv")
                  .filter(clean_mask)
                  .sum()
                  .alias(f"gmv_clean_sum_{w}"),
                pl.col("to_ord")
                  .filter(clean_mask)
                  .sum()
                  .alias(f"to_ord_clean_sum_{w}"),
                pl.col("searches")
                  .filter(clean_mask)
                  .sum()
                  .alias(f"searches_clean_sum_{w}"),
                pl.col("event_date")
                  .filter(clean_mask)
                  .count()
                  .alias(f"active_days_clean_{w}"),
            ])

        if w in NEW_YEAR_WINDOWS:
            ny_mask = mask & pl.col("is_newyear_window")
            exprs.extend([
                pl.col("gmv")
                  .filter(ny_mask)
                  .sum()
                  .alias(f"gmv_newyear_sum_{w}"),
                pl.col("to_ord")
                  .filter(ny_mask)
                  .sum()
                  .alias(f"to_ord_newyear_sum_{w}"),
                pl.col("event_date")
                  .filter(ny_mask)
                  .count()
                  .alias(f"active_days_newyear_{w}"),
            ])

        if w in [90, 180, 365]:
            weekend_mask = mask & pl.col("event_date").dt.weekday().is_in([6, 7])
            exprs.extend([
                pl.col("gmv")
                  .filter(weekend_mask)
                  .sum()
                  .alias(f"gmv_weekend_sum_{w}"),
                pl.col("to_ord")
                  .filter(weekend_mask)
                  .sum()
                  .alias(f"to_ord_weekend_sum_{w}"),
            ])

    exprs.extend([
        _date_diff_expr(anchor, pl.lit(True), "recency_active"),
        _date_diff_expr(anchor, pl.col("to_cart") > 0, "recency_cart"),
        _date_diff_expr(anchor, pl.col("to_ord") > 0, "recency_order"),
        _date_diff_expr(anchor, pl.col("gmv") > 0, "recency_gmv"),
    ])

    for w in SPAN_WINDOWS:
        start = anchor - timedelta(days=w - 1)
        mask = pl.col("event_date") >= pl.lit(start)

        exprs.extend([
            pl.col("event_date")
              .filter(mask)
              .cast(pl.Int32)
              .min()
              .alias(f"_first_active_day_{w}"),
            pl.col("event_date")
              .filter(mask)
              .cast(pl.Int32)
              .max()
              .alias(f"_last_active_day_{w}"),
            pl.col("event_date")
              .filter(mask & (pl.col("to_ord") > 0))
              .cast(pl.Int32)
              .min()
              .alias(f"_first_order_day_{w}"),
            pl.col("event_date")
              .filter(mask & (pl.col("to_ord") > 0))
              .cast(pl.Int32)
              .max()
              .alias(f"_last_order_day_{w}"),
        ])

    agg = hist.group_by("user_id").agg(exprs)
    feat = all_users.join(agg, on="user_id", how="left")

    span_exprs: list[pl.Expr] = []
    span_drop: list[str] = []

    for w in SPAN_WINDOWS:
        span_exprs.extend([
            (
                pl.col(f"_last_active_day_{w}")
                - pl.col(f"_first_active_day_{w}")
                + 1
            ).fill_null(0).alias(f"active_span_{w}"),
            (
                pl.col(f"_last_order_day_{w}")
                - pl.col(f"_first_order_day_{w}")
                + 1
            ).fill_null(0).alias(f"order_span_{w}"),
        ])
        span_drop.extend([
            f"_first_active_day_{w}",
            f"_last_active_day_{w}",
            f"_first_order_day_{w}",
            f"_last_order_day_{w}",
        ])

    feat = feat.with_columns(span_exprs).drop(span_drop)

    recency_cols = [
        "recency_active", "recency_cart", "recency_order", "recency_gmv"
    ]

    numeric_cols = [c for c in feat.columns if c != "user_id"]

    feat = feat.with_columns([
        pl.col(c).fill_null(RECENCY_SENTINEL).cast(pl.Float32)
        if c in recency_cols
        else pl.col(c).fill_null(0).cast(pl.Float32)
        for c in numeric_cols
    ])

    derived: list[pl.Expr] = []

    for w in WINDOWS:
        observed_days = available_calendar_days(anchor, w)

        derived.extend([
            pl.lit(float(observed_days)).alias(f"observed_calendar_days_{w}"),
            pl.lit(float(observed_days) / float(w))
              .alias(f"history_coverage_{w}"),

            (pl.col(f"gmv_sum_{w}") / (pl.col(f"to_ord_sum_{w}") + 1.0))
              .alias(f"aov_{w}"),

            (pl.col(f"to_cart_sum_{w}") / (pl.col(f"searches_sum_{w}") + 1.0))
              .alias(f"cart_per_search_{w}"),

            (pl.col(f"to_ord_sum_{w}") / (pl.col(f"to_cart_sum_{w}") + 1.0))
              .alias(f"ord_per_cart_{w}"),

            (pl.col(f"gmv_search_sum_{w}") / (pl.col(f"gmv_sum_{w}") + 1.0))
              .alias(f"gmv_search_share_{w}"),

            (pl.col(f"search_to_ord_sum_{w}") / (pl.col(f"to_ord_sum_{w}") + 1.0))
              .alias(f"order_search_share_{w}"),

            (pl.col(f"active_days_{w}") / float(observed_days))
              .alias(f"active_rate_{w}"),

            (pl.col(f"gmv_sum_{w}") * (30.0 / float(observed_days)))
              .alias(f"gmv_runrate_{w}"),

            (pl.col(f"to_ord_sum_{w}") * (30.0 / float(observed_days)))
              .alias(f"orders_runrate_{w}"),

            (pl.col(f"searches_sum_{w}") * (30.0 / float(observed_days)))
              .alias(f"searches_runrate_{w}"),
        ])

    feat = feat.with_columns(derived)

    prev_7_gmv = pl.col("gmv_sum_14") - pl.col("gmv_sum_7")
    prev_30_gmv = pl.col("gmv_sum_60") - pl.col("gmv_sum_30")
    prev_30_ord = pl.col("to_ord_sum_60") - pl.col("to_ord_sum_30")
    prev_30_search = pl.col("searches_sum_60") - pl.col("searches_sum_30")

    prev_90_gmv = pl.col("gmv_sum_180") - pl.col("gmv_sum_90")
    prev_90_ord = pl.col("to_ord_sum_180") - pl.col("to_ord_sum_90")

    prev_180_gmv = pl.col("gmv_sum_365") - pl.col("gmv_sum_180")
    prev_180_ord = pl.col("to_ord_sum_365") - pl.col("to_ord_sum_180")

    feat = feat.with_columns([
        ((pl.col("gmv_sum_7") + 1.0) / (prev_7_gmv + 1.0))
            .clip(0, 50).alias("gmv_trend_7_vs_prev7"),

        ((pl.col("gmv_sum_30") + 1.0) / (prev_30_gmv + 1.0))
            .clip(0, 50).alias("gmv_trend_30_vs_prev30"),

        ((pl.col("to_ord_sum_30") + 1.0) / (prev_30_ord + 1.0))
            .clip(0, 50).alias("ord_trend_30_vs_prev30"),

        ((pl.col("searches_sum_30") + 1.0) / (prev_30_search + 1.0))
            .clip(0, 50).alias("search_trend_30_vs_prev30"),

        ((pl.col("gmv_sum_90") + 1.0) / (prev_90_gmv + 1.0))
            .clip(0, 50).alias("gmv_trend_90_vs_prev90"),

        ((pl.col("to_ord_sum_90") + 1.0) / (prev_90_ord + 1.0))
            .clip(0, 50).alias("ord_trend_90_vs_prev90"),

        ((pl.col("gmv_sum_180") + 1.0) / (prev_180_gmv + 1.0))
            .clip(0, 50).alias("gmv_trend_180_vs_prev180"),

        ((pl.col("to_ord_sum_180") + 1.0) / (prev_180_ord + 1.0))
            .clip(0, 50).alias("ord_trend_180_vs_prev180"),

        (pl.col("gmv_sum_30") / (pl.col("gmv_sum_180") + 1.0))
            .clip(0, 1).alias("gmv_recent_share_30_180"),

        (pl.col("gmv_sum_30") / (pl.col("gmv_sum_365") + 1.0))
            .clip(0, 1).alias("gmv_recent_share_30_365"),

        (pl.col("gmv_sum_90") / (pl.col("gmv_sum_365") + 1.0))
            .clip(0, 1).alias("gmv_recent_share_90_365"),

        (pl.col("gmv_sum_7") * (30.0 / 7.0)).alias("gmv_runrate_7_raw"),
        (pl.col("gmv_sum_14") * (30.0 / 14.0)).alias("gmv_runrate_14_raw"),
    ])

    feat = feat.with_columns([
        (
            pl.col("search_to_cart_sum_90")
            / pl.col("searches_sum_90").clip(lower_bound=1)
        ).alias("search_to_cart_rate"),

        (
            pl.col("search_to_ord_sum_90")
            / pl.col("searches_sum_90").clip(lower_bound=1)
        ).alias("search_to_order_rate"),

        (
            pl.col("cat_to_cart_sum_90")
            / pl.col("cat_sum_90").clip(lower_bound=1)
        ).alias("cat_to_cart_rate"),

        (
            pl.col("searches_sum_90")
            / pl.col("active_days_90").clip(lower_bound=1)
        ).alias("searches_per_active_day"),

        pl.col("recency_active").alias("recency_days"),
        pl.col("active_days_90").alias("active_days"),

        (
            pl.col("to_ord_sum_30")
            / pl.col("to_ord_sum_90").clip(lower_bound=1)
        ).alias("orders_recent_share"),
    ])

    stability_exprs: list[pl.Expr] = []

    for w in SPAN_WINDOWS:
        stability_exprs.extend([
            (
                pl.col(f"active_days_{w}")
                / pl.col(f"active_span_{w}").clip(lower_bound=1)
            ).alias(f"active_density_span_{w}"),

            (
                pl.col(f"order_days_{w}")
                / pl.col(f"order_span_{w}").clip(lower_bound=1)
            ).alias(f"order_density_span_{w}"),

            (
                pl.col(f"to_ord_sum_{w}")
                / pl.col(f"order_days_{w}").clip(lower_bound=1)
            ).alias(f"orders_per_order_day_{w}"),

            (
                pl.col(f"gmv_sum_{w}")
                / pl.col(f"order_days_{w}").clip(lower_bound=1)
            ).alias(f"gmv_per_order_day_{w}"),

            (pl.col(f"to_ord_sum_{w}") > 0)
              .cast(pl.Float32)
              .alias(f"has_order_{w}"),
        ])

    for w in WEEK_WINDOWS:
        stability_exprs.extend([
            (
                pl.col(f"active_weeks_{w}")
                / max(1.0, available_calendar_days(anchor, w) / 7.0)
            ).clip(0, 1).alias(f"active_week_density_{w}"),

            (
                pl.col(f"order_weeks_{w}")
                / max(1.0, available_calendar_days(anchor, w) / 7.0)
            ).clip(0, 1).alias(f"order_week_density_{w}"),
        ])

    feat = feat.with_columns(stability_exprs)

    weekend_exprs: list[pl.Expr] = []
    for w in [90, 180, 365]:
        weekend_exprs.extend([
            (
                pl.col(f"gmv_weekend_sum_{w}")
                / (pl.col(f"gmv_sum_{w}") + 1.0)
            ).clip(0, 1).alias(f"weekend_gmv_share_{w}"),

            (
                pl.col(f"to_ord_weekend_sum_{w}")
                / (pl.col(f"to_ord_sum_{w}") + 1.0)
            ).clip(0, 1).alias(f"weekend_order_share_{w}"),
        ])
    feat = feat.with_columns(weekend_exprs)

    special_derived: list[pl.Expr] = []

    for w in CLEAN_WINDOWS:
        actual_start = max(min_date, anchor - timedelta(days=w - 1))
        observed_days = max(1, (anchor - actual_start).days + 1)

        special_days = count_special_days(actual_start, anchor)
        clean_calendar_days = max(1, observed_days - special_days)
        newyear_days = count_newyear_days(actual_start, anchor)

        special_derived.extend([
            pl.lit(float(special_days)).alias(f"special_calendar_days_{w}"),
            pl.lit(float(newyear_days)).alias(f"newyear_calendar_days_{w}"),

            (
                pl.col(f"gmv_clean_sum_{w}")
                * (30.0 / float(clean_calendar_days))
            ).alias(f"gmv_clean_runrate_{w}"),

            (
                pl.col(f"to_ord_clean_sum_{w}")
                * (30.0 / float(clean_calendar_days))
            ).alias(f"orders_clean_runrate_{w}"),

            (
                (
                    pl.col(f"gmv_sum_{w}")
                    * (30.0 / float(observed_days))
                )
                / (
                    pl.col(f"gmv_clean_sum_{w}")
                    * (30.0 / float(clean_calendar_days))
                    + 1.0
                )
            ).clip(0, 20).alias(f"gmv_raw_to_clean_ratio_{w}"),

            (
                pl.col(f"searches_clean_sum_{w}")
                * (30.0 / float(clean_calendar_days))
            ).alias(f"searches_clean_runrate_{w}"),

            (
                (pl.col(f"gmv_sum_{w}") - pl.col(f"gmv_clean_sum_{w}"))
                / (pl.col(f"gmv_sum_{w}") + 1.0)
            ).clip(0, 1).alias(f"special_gmv_share_{w}"),

            (
                pl.col(f"gmv_newyear_sum_{w}")
                / (pl.col(f"gmv_sum_{w}") + 1.0)
            ).clip(0, 1).alias(f"newyear_gmv_share_{w}"),

            (
                pl.col(f"to_ord_newyear_sum_{w}")
                / (pl.col(f"to_ord_sum_{w}") + 1.0)
            ).clip(0, 1).alias(f"newyear_order_share_{w}"),

            (
                pl.col(f"gmv_newyear_sum_{w}")
                / max(1.0, float(newyear_days))
            ).alias(f"newyear_gmv_per_calendar_day_{w}"),
        ])

    feat = feat.with_columns(special_derived)

    global_values = global_state_values(anchor)
    cal = future_calendar_values(anchor)

    history_days_available = min(
        max(WINDOWS),
        (anchor - min_date).days + 1,
    )

    feat = feat.with_columns([
        pl.lit(anchor).cast(pl.Date).alias("anchor_date"),
        pl.lit(anchor.month).cast(pl.Float32).alias("anchor_month"),
        pl.lit(anchor.weekday()).cast(pl.Float32).alias("anchor_weekday"),
        pl.lit(anchor.timetuple().tm_yday).cast(pl.Float32).alias("anchor_doy"),
        pl.lit(float(history_days_available))
          .cast(pl.Float32)
          .alias("history_days_available"),
        *[
            pl.lit(v).cast(pl.Float32).alias(k)
            for k, v in {**cal, **global_values}.items()
        ],
    ])

    log_candidates = [
        c for c in feat.columns
        if (
            c.startswith("gmv_sum_")
            or c.startswith("to_ord_sum_")
            or c.startswith("searches_sum_")
            or c.startswith("gmv_runrate_")
            or c.startswith("gmv_clean_runrate_")
            or c.startswith("orders_runrate_")
            or c.startswith("order_days_")
        )
        and c not in ["anchor_date"]
    ]

    feat = feat.with_columns([
        pl.col(c)
          .clip(lower_bound=0)
          .log1p()
          .alias(f"log1p_{c}")
        for c in log_candidates
    ])

    to_float = [
        c for c in feat.columns
        if c not in ["user_id", "anchor_date"]
    ]

    feat = feat.with_columns([
        pl.col(c)
          .fill_nan(0)
          .fill_null(0)
          .cast(pl.Float32)
        for c in to_float
    ])

    return feat

def make_target_for_anchor(
    data: pl.DataFrame,
    all_users: pl.DataFrame,
    anchor: date,
) -> pl.DataFrame:
    start = anchor + timedelta(days=1)
    end = anchor + timedelta(days=HORIZON)

    tgt = (
        data.filter(pl.col("event_date").is_between(start, end, closed="both"))
        .group_by("user_id")
        .agg(pl.col("gmv").sum().alias("target"))
    )

    return (
        all_users
        .join(tgt, on="user_id", how="left")
        .with_columns(
            pl.col("target")
              .fill_null(0)
              .clip(lower_bound=0)
              .cast(pl.Float32)
        )
    )

def build_anchor_frame(anchor: date, with_target: bool = True) -> pl.DataFrame:
    suffix = "train" if with_target else "infer"
    path = CACHE_DIR / f"{anchor.isoformat()}_{suffix}.parquet"

    if path.exists():
        return pl.read_parquet(path)

    feat = make_features_for_anchor(data, all_users, anchor)

    if with_target:
        tgt = make_target_for_anchor(data, all_users, anchor)
        feat = (
            feat.join(tgt, on="user_id", how="left")
            .with_columns(
                pl.col("target")
                  .fill_null(0)
                  .clip(lower_bound=0)
                  .cast(pl.Float32)
            )
        )

    feat.write_parquet(path, compression="zstd")
    return feat


In [6]:
C_FEATURES = [
    "active_days_420",
    "order_days_420",
    "gmv_sum_420",
    "to_ord_sum_420",
    "searches_sum_420",
    "gmv_clean_sum_420",
    "to_ord_clean_sum_420",
    "searches_clean_sum_420",
    "history_coverage_420",
    "active_week_density_420",
    "order_week_density_420",
    "gmv_runrate_420",
    "orders_runrate_420",
    "searches_runrate_420",
    "gmv_clean_runrate_420",
    "orders_clean_runrate_420",
    "searches_clean_runrate_420",
    "aov_420",
    "purchase_intensity_420",
    "orders_per_order_day_420",
    "gmv_per_order_day_420",
    "log1p_order_days_420",
    "log1p_to_ord_sum_420",
    "log1p_gmv_sum_420",
    "log1p_searches_sum_420",
    "log1p_gmv_runrate_420",
    "log1p_orders_runrate_420",
    "log1p_gmv_clean_runrate_420",
]

NY_WINDOWS_RESCUE = [30, 60, 90, 180, 270, 365]
D_FEATURES = (
    [f"gmv_newyear_{part}_sum_{w}" for w in NY_WINDOWS_RESCUE for part in ["early", "peak", "post"]]
    + [f"newyear_{part}_gmv_share_{w}" for w in NY_WINDOWS_RESCUE for part in ["early", "peak", "post"]]
    + ["target_newyear_early_days", "target_newyear_peak_days"]
)

assert len(C_FEATURES) == 28
assert len(D_FEATURES) == 38
assert len(set(C_FEATURES + D_FEATURES)) == 66

def _extra_cache_path(candidate: str, anchor: date) -> Path:
    d = CACHE_EXTRA_DIR / candidate
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{anchor.isoformat()}.parquet"

def _finalize_extra(frame: pl.DataFrame, feature_cols: list[str]) -> pl.DataFrame:
    missing = [c for c in feature_cols if c not in frame.columns]
    assert not missing, f"Missing extra columns: {missing}"
    out = frame.select(["user_id", *feature_cols]).with_columns([
        pl.col(c).fill_nan(0).fill_null(0).cast(pl.Float32)
        for c in feature_cols
    ])
    assert out.height == all_users.height
    assert out["user_id"].n_unique() == all_users.height
    return out

def _read_cached_extra(path: Path, expected: list[str]) -> pl.DataFrame | None:
    if not path.exists() or FORCE_REBUILD_EXTRA:
        return None
    out = pl.read_parquet(path)
    assert out.columns == ["user_id", *expected], (path, out.columns[:8], len(out.columns))
    assert out.height == all_users.height
    return out

def build_C_extra(anchor: date) -> pl.DataFrame:
    cp = _extra_cache_path("C_compact_fullhistory", anchor)
    cached = _read_cached_extra(cp, C_FEATURES)
    if cached is not None:
        return cached

    w = 420
    start = max(min_date, anchor - timedelta(days=w - 1))
    hist = data.filter(pl.col("event_date").is_between(start, anchor, closed="both"))
    clean_mask = ~pl.col("is_special_day")

    agg = hist.group_by("user_id").agg([
        pl.col("event_date").count().alias("active_days_420"),
        pl.col("event_date").filter(pl.col("to_ord") > 0).count().alias("order_days_420"),
        pl.col("event_date").dt.truncate("1w").n_unique().alias("active_weeks_420"),
        pl.col("event_date").filter(pl.col("to_ord") > 0).dt.truncate("1w").n_unique().alias("order_weeks_420"),
        pl.col("gmv").sum().alias("gmv_sum_420"),
        pl.col("to_ord").sum().alias("to_ord_sum_420"),
        pl.col("searches").sum().alias("searches_sum_420"),
        pl.col("gmv").filter(clean_mask).sum().alias("gmv_clean_sum_420"),
        pl.col("to_ord").filter(clean_mask).sum().alias("to_ord_clean_sum_420"),
        pl.col("searches").filter(clean_mask).sum().alias("searches_clean_sum_420"),
    ])

    out = all_users.join(agg, on="user_id", how="left")
    raw_cols = [c for c in out.columns if c != "user_id"]
    out = out.with_columns([pl.col(c).fill_null(0).cast(pl.Float32) for c in raw_cols])

    observed_days = available_calendar_days(anchor, 420)
    special_days = count_special_days(start, anchor)
    clean_calendar_days = max(1, observed_days - special_days)

    out = out.with_columns([
        pl.lit(float(observed_days) / 420.0).alias("history_coverage_420"),
        (pl.col("active_weeks_420") / max(1.0, observed_days / 7.0)).clip(0, 1).alias("active_week_density_420"),
        (pl.col("order_weeks_420") / max(1.0, observed_days / 7.0)).clip(0, 1).alias("order_week_density_420"),
        (pl.col("gmv_sum_420") * (30.0 / observed_days)).alias("gmv_runrate_420"),
        (pl.col("to_ord_sum_420") * (30.0 / observed_days)).alias("orders_runrate_420"),
        (pl.col("searches_sum_420") * (30.0 / observed_days)).alias("searches_runrate_420"),
        (pl.col("gmv_clean_sum_420") * (30.0 / clean_calendar_days)).alias("gmv_clean_runrate_420"),
        (pl.col("to_ord_clean_sum_420") * (30.0 / clean_calendar_days)).alias("orders_clean_runrate_420"),
        (pl.col("searches_clean_sum_420") * (30.0 / clean_calendar_days)).alias("searches_clean_runrate_420"),
        (pl.col("gmv_sum_420") / (pl.col("to_ord_sum_420") + 1.0)).alias("aov_420"),
        (pl.col("to_ord_sum_420") / pl.col("searches_sum_420").clip(lower_bound=1)).alias("purchase_intensity_420"),
        (pl.col("to_ord_sum_420") / pl.col("order_days_420").clip(lower_bound=1)).alias("orders_per_order_day_420"),
        (pl.col("gmv_sum_420") / pl.col("order_days_420").clip(lower_bound=1)).alias("gmv_per_order_day_420"),
    ])

    for raw in [
        "order_days_420", "to_ord_sum_420", "gmv_sum_420", "searches_sum_420",
        "gmv_runrate_420", "orders_runrate_420", "gmv_clean_runrate_420",
    ]:
        out = out.with_columns(pl.col(raw).clip(lower_bound=0).log1p().alias(f"log1p_{raw}"))

    out = _finalize_extra(out, C_FEATURES)
    out.write_parquet(cp, compression="zstd")
    return out

def is_newyear_early_day_cd(d: date) -> bool:
    return d.month == 12 and 15 <= d.day <= 21

def is_newyear_peak_day_cd(d: date) -> bool:
    return d.month == 12 and d.day >= 22

def future_detailed_ny_values(anchor: date) -> dict[str, float]:
    days = [anchor + timedelta(days=i) for i in range(1, HORIZON + 1)]
    return {
        "target_newyear_early_days": float(sum(is_newyear_early_day_cd(d) for d in days)),
        "target_newyear_peak_days": float(sum(is_newyear_peak_day_cd(d) for d in days)),
    }

def build_D_extra(anchor: date, base_frame: pl.DataFrame) -> pl.DataFrame:
    cp = _extra_cache_path("D_detailed_newyear", anchor)
    cached = _read_cached_extra(cp, D_FEATURES)
    if cached is not None:
        return cached

    hist_start = max(min_date, anchor - timedelta(days=max(NY_WINDOWS_RESCUE) - 1))
    hist = data.filter(pl.col("event_date").is_between(hist_start, anchor, closed="both"))

    month = pl.col("event_date").dt.month().cast(pl.Int32)
    day = pl.col("event_date").dt.day().cast(pl.Int32)
    early = (month == 12) & (day >= 15) & (day <= 21)
    peak = (month == 12) & (day >= 22)
    post = (month == 1) & (day <= 3)

    exprs: list[pl.Expr] = []
    for w in NY_WINDOWS_RESCUE:
        start = anchor - timedelta(days=w - 1)
        in_w = pl.col("event_date") >= pl.lit(start)
        exprs.extend([
            pl.col("gmv").filter(in_w & early).sum().alias(f"gmv_newyear_early_sum_{w}"),
            pl.col("gmv").filter(in_w & peak).sum().alias(f"gmv_newyear_peak_sum_{w}"),
            pl.col("gmv").filter(in_w & post).sum().alias(f"gmv_newyear_post_sum_{w}"),
        ])

    raw = hist.group_by("user_id").agg(exprs)
    out = all_users.join(raw, on="user_id", how="left")
    raw_ny_cols = [c for c in out.columns if c != "user_id"]
    out = out.with_columns([pl.col(c).fill_null(0).cast(pl.Float32) for c in raw_ny_cols])

    gmv_cols = [f"gmv_sum_{w}" for w in NY_WINDOWS_RESCUE]
    out = out.join(base_frame.select(["user_id", *gmv_cols]), on="user_id", how="left")

    share_exprs: list[pl.Expr] = []
    for w in NY_WINDOWS_RESCUE:
        denom = pl.col(f"gmv_sum_{w}") + 1.0
        share_exprs.extend([
            (pl.col(f"gmv_newyear_early_sum_{w}") / denom).clip(0, 1).alias(f"newyear_early_gmv_share_{w}"),
            (pl.col(f"gmv_newyear_peak_sum_{w}") / denom).clip(0, 1).alias(f"newyear_peak_gmv_share_{w}"),
            (pl.col(f"gmv_newyear_post_sum_{w}") / denom).clip(0, 1).alias(f"newyear_post_gmv_share_{w}"),
        ])
    out = out.with_columns(share_exprs)

    cal = future_detailed_ny_values(anchor)
    out = out.with_columns([pl.lit(v).cast(pl.Float32).alias(k) for k, v in cal.items()])

    out = _finalize_extra(out, D_FEATURES)
    out.write_parquet(cp, compression="zstd")
    return out

def build_cd_frame(anchor: date, with_target: bool = True) -> pl.DataFrame:
    base = build_anchor_frame(anchor, with_target=with_target)
    c = build_C_extra(anchor)
    d = build_D_extra(anchor, base)
    out = base.join(c, on="user_id", how="left").join(d, on="user_id", how="left")
    assert out.height == base.height
    return out


In [7]:
_probe = build_cd_frame(date(2026, 1, 14), with_target=True)
DROP_COLS = {"user_id", "anchor_date", "target"}
BASE_FEATURE_COLS = [c for c in _probe.columns if c not in DROP_COLS]

assert len(BASE_FEATURE_COLS) == 591
assert len(BASE_FEATURE_COLS) == len(set(BASE_FEATURE_COLS))

print("Base features:", len(BASE_FEATURE_COLS))

del _probe
gc.collect()


Base features: 591


0

In [14]:
V4_NULLABLE_FEATURE_COLS = {
    c for c in V4_FEATURE_COLS
    if c.startswith("current_arch5_w_")
    or c.startswith("future_arch5_delta_w_")
    or c in {
        "current_arch5_dominant",
        "current_arch5_dominant_weight",
        "current_arch5_entropy",
        "current_arch5_effective_count",
        "future_vs_current_arch5_cosine",
        "future_vs_current_arch5_l1",
        "future_vs_current_arch5_dominant_same",
    }
}


def _audit_temporal_frame(
    df: pl.DataFrame,
    anchor: date,
    label: str,
) -> None:
    assert df.height == EXPECTED_USERS, (label, anchor, df.shape)
    assert df["user_id"].n_unique() == EXPECTED_USERS
    assert df["anchor_date"].unique().to_list() == [anchor]

    sums = df.select(
        pl.sum_horizontal(GATE_COLS).alias("s")
    )["s"].to_numpy()

    assert np.isfinite(sums).all()
    assert np.allclose(sums, 1.0, atol=2e-4)

    q = df[QUALITY_COL].to_numpy().astype(
        np.float64,
        copy=False,
    )

    assert np.isfinite(q).all()
    assert float(q.min()) >= -1e-6
    assert float(q.max()) <= 1.0 + 1e-6

    assert df.filter(
        pl.col("aux_fit_cutoff") > pl.col("anchor_date")
    ).height == 0

    assert df.filter(
        pl.col("arch_fit_cutoff") > pl.col("anchor_date")
    ).height == 0

    for c in V4_FEATURE_COLS:
        if c in V4_NULLABLE_FEATURE_COLS:
            arr = df[c].drop_nulls().to_numpy()
        else:
            arr = df[c].to_numpy()

        assert np.isfinite(arr).all(), (anchor, c)


def temporal_anchor_source(
    anchor: date,
) -> pl.DataFrame:
    if anchor == max_date:
        src = TEMP_FINAL
    else:
        src = (
            pl.scan_parquet(TEMP_HIST_PATH)
            .select(TEMP_READ_COLS)
            .with_columns([
                pl.col("anchor_date").cast(pl.Date),
                pl.col("aux_fit_cutoff").cast(pl.Date),
                pl.col("arch_fit_cutoff").cast(pl.Date),
            ])
            .filter(
                pl.col("anchor_date") == pl.lit(anchor)
            )
            .collect()
        )

    _audit_temporal_frame(
        src,
        anchor,
        "V4",
    )

    return src


def _attach_columns_1to1(
    frame: pl.DataFrame,
    src: pl.DataFrame,
    cols: list[str],
) -> pl.DataFrame:
    src2 = src.select([
        "user_id",
        "anchor_date",
        *cols,
    ])

    before = frame.height

    out = (
        frame
        .with_row_index("__row_order")
        .join(
            src2,
            on=["user_id", "anchor_date"],
            how="left",
            validate="m:1",
        )
        .sort("__row_order")
        .drop("__row_order")
    )

    assert out.height == before

    for c in cols:
        assert c in out.columns

        if c not in V4_NULLABLE_FEATURE_COLS:
            assert out[c].null_count() == 0, c

    return out


def attach_v4(
    frame: pl.DataFrame,
    anchor: date,
) -> pl.DataFrame:
    src = temporal_anchor_source(anchor)

    cols = list(dict.fromkeys([
        "aux_fit_cutoff",
        "arch_fit_cutoff",
        *V4_FEATURE_COLS,
    ]))

    return _attach_columns_1to1(
        frame,
        src,
        cols,
    )

## Модель

In [9]:
def recent_params():
    return dict(
        iterations=CAPACITY["recent"],
        loss_function="RMSE", eval_metric="RMSE",
        depth=7,
        learning_rate=0.040,
        l2_leaf_reg=12.0,
        random_strength=0.30,
        bootstrap_type="Bayesian",
        bagging_temperature=0.75,
        border_count=BORDER_COUNT,
        random_seed=SEED_RECENT,
        allow_writing_files=False,
        task_type="GPU",
        devices=DEVICES,
        verbose=200,
    )

def classifier_params():
    return dict(
        iterations=CAPACITY["classifier"],
        loss_function="Logloss", eval_metric="Logloss",
        depth=7,
        learning_rate=0.04,
        l2_leaf_reg=8.0,
        random_strength=0.30,
        bootstrap_type="Bayesian",
        bagging_temperature=0.7,
        border_count=BORDER_COUNT,
        random_seed=SEED_CLASSIFIER,
        allow_writing_files=False,
        task_type="GPU",
        devices=DEVICES,
        verbose=200,
    )

def positive_params():
    return dict(
        iterations=CAPACITY["positive"],
        loss_function="RMSE", eval_metric="RMSE",
        depth=8,
        learning_rate=0.032,
        l2_leaf_reg=10.0,
        random_strength=0.22,
        bootstrap_type="Bayesian",
        bagging_temperature=0.6,
        border_count=BORDER_COUNT,
        random_seed=SEED_POSITIVE,
        allow_writing_files=False,
        task_type="GPU",
        devices=DEVICES,
        verbose=200,
    )

print("recent:", recent_params())
print("classifier:", classifier_params())
print("positive:", positive_params())


recent: {'iterations': 1138, 'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'depth': 7, 'learning_rate': 0.04, 'l2_leaf_reg': 12.0, 'random_strength': 0.3, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.75, 'border_count': 254, 'random_seed': 143, 'allow_writing_files': False, 'task_type': 'GPU', 'devices': '0', 'verbose': 200}
classifier: {'iterations': 820, 'loss_function': 'Logloss', 'eval_metric': 'Logloss', 'depth': 7, 'learning_rate': 0.04, 'l2_leaf_reg': 8.0, 'random_strength': 0.3, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.7, 'border_count': 254, 'random_seed': 243, 'allow_writing_files': False, 'task_type': 'GPU', 'devices': '0', 'verbose': 200}
positive: {'iterations': 845, 'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'depth': 8, 'learning_rate': 0.032, 'l2_leaf_reg': 10.0, 'random_strength': 0.22, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 0.6, 'border_count': 254, 'random_seed': 343, 'allow_writing_files': False, 'task_type': 'GPU', 'devic

In [10]:
def _eligible_train_anchors_strict(outer_anchor: date) -> list[date]:
    return [
        a for a in V9_CANONICAL_POOL
        if a != outer_anchor and a + timedelta(days=HORIZON) <= outer_anchor
    ]

def effective_n(w: np.ndarray) -> float:
    w = np.asarray(w, dtype=np.float64)
    w = w[np.isfinite(w) & (w > 0)]
    if len(w) == 0:
        return 0.0
    return float((w.sum() ** 2) / np.sum(w * w))

def _filter_mask(frame: pl.DataFrame, mask: np.ndarray) -> pl.DataFrame:
    mask = np.asarray(mask, dtype=bool)
    assert len(mask) == frame.height
    return frame.filter(pl.Series('__mask', mask))

def _matrix(frame: pl.DataFrame, cols: list[str], mask: np.ndarray | None = None) -> np.ndarray:
    part = frame if mask is None else _filter_mask(frame, mask)
    return part.select(cols).to_numpy().astype(np.float32, copy=False)

def _append_importance(rows: list[dict], model, cols: list[str], fold: str, branch: str, expert: str):
    fi = np.asarray(model.get_feature_importance(), dtype=np.float64)
    order = np.argsort(-fi)
    for rank, j in enumerate(order, 1):
        rows.append({
            'fold': fold,
            'branch': branch,
            'expert': expert,
            'feature': cols[j],
            'importance': float(fi[j]),
            'rank': rank,
            'is_latent': bool(cols[j].startswith('gru_latent_') or cols[j].startswith('tcn_latent_')),
        })

def make_future_component_targets(anchor: date) -> pl.DataFrame:
    cp = V12_TARGET_CACHE_DIR / f'future_components_{anchor}.parquet'
    if cp.exists():
        return pl.read_parquet(cp)

    start = anchor + timedelta(days=1)
    end = anchor + timedelta(days=HORIZON)
    agg = (
        data.filter(pl.col('event_date').is_between(start, end, closed='both'))
        .group_by('user_id')
        .agg([
            pl.col('gmv').sum().alias('__target_check_raw'),
            (pl.col('gmv') > 0).sum().alias('future_spend_days'),
            pl.col('to_ord').sum().alias('future_orders_raw'),
            (pl.col('to_ord') > 0).sum().alias('future_order_days_raw'),
        ])
    )
    out = (
        all_users.join(agg, on='user_id', how='left')
        .with_columns([
            pl.col('__target_check_raw').fill_null(0).clip(lower_bound=0).cast(pl.Float32),
            pl.col('future_spend_days').fill_null(0).cast(pl.Float32),
            pl.col('future_orders_raw').fill_null(0).clip(lower_bound=0).cast(pl.Float32),
            pl.col('future_order_days_raw').fill_null(0).cast(pl.Float32),
        ])
    )
    out.write_parquet(cp, compression='zstd', statistics=True)
    return out

def build_v12_frame(anchor: date, with_target: bool = True) -> pl.DataFrame:
    base = build_model_frame(anchor, with_target=with_target)
    if not with_target:
        return base
    comp = make_future_component_targets(anchor)
    out = (
        base.with_row_index('__row_order')
        .join(comp, on='user_id', how='left', validate='m:1')
        .sort('__row_order')
        .drop('__row_order')
    )
    assert out.height == base.height
    diff = np.abs(
        out['target'].to_numpy().astype(np.float64)
        - out['__target_check_raw'].to_numpy().astype(np.float64)
    )
    assert float(diff.max()) < 1e-3, (anchor, float(diff.max()))
    y = out['target'].to_numpy().astype(np.float64)
    spend_days = out['future_spend_days'].to_numpy().astype(np.float64)
    impossible = (y > 0) & (spend_days <= 0)
    assert not np.any(impossible), (anchor, int(impossible.sum()))
    out = out.with_columns(
        pl.when(pl.col('target') > 0)
        .then(pl.col('target') / pl.col('future_spend_days').clip(lower_bound=1.0))
        .otherwise(0.0)
        .cast(pl.Float32)
        .alias('future_spend_per_day')
    )
    return out

BASE_VALUE_FEATURE_COLS = list(dict.fromkeys(BASE_FEATURE_COLS + V4_FEATURE_COLS))
assert len(BASE_VALUE_FEATURE_COLS) == len(set(BASE_VALUE_FEATURE_COLS))
print('FI value feature count:', len(BASE_VALUE_FEATURE_COLS))

def file_sha256(path: Path | None, chunk_size: int = 1 << 20) -> str | None:
    if path is None or not Path(path).exists():
        return None
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def json_sha256(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


FI value feature count: 662


In [11]:
def global_feature_spec(mode: str) -> dict[str, list[str]]:
    if mode == "v9_control":
        return {
            "recent": BASE_FEATURE_COLS + RECENT_AUX,
            "occurrence": BASE_FEATURE_COLS + HURDLE_AUX,
            "positive": BASE_FEATURE_COLS,
        }
    raise ValueError(mode)

def fit_predict_v9_control(
    full: pl.DataFrame,
    val: pl.DataFrame,
    eligible_anchors: list[date],
    fold_label: str,
    importance_rows: list[dict],
) -> dict[str, np.ndarray]:
    spec = global_feature_spec("v9_control")
    recent_anchors = sorted(eligible_anchors)[-RECENT_MODEL_N_ANCHORS:]
    recent = full.filter(pl.col("anchor_date").is_in(recent_anchors))

    y_full = full["target"].to_numpy().astype(np.float64, copy=False)
    ylog_full = np.log1p(np.clip(y_full, 0, None))
    pos = y_full > 0
    sw_full = recency_weights(full, decay_days=RECENCY_DECAY_ATTR)

    yr = recent["target"].to_numpy().astype(np.float64, copy=False)
    yrlog = np.log1p(np.clip(yr, 0, None))
    sw_recent = temporal_weights_against_anchor_pool(recent, eligible_anchors, decay_days=RECENCY_DECAY_ATTR)
    Xtr = _matrix(recent, spec["recent"])
    Xva = _matrix(val, spec["recent"])
    recent_model = CatBoostRegressor(**recent_params())
    recent_model.fit(Xtr, yrlog, sample_weight=sw_recent, verbose=200)
    pred_recent = np.clip(recent_model.predict(Xva), 0, None).astype(np.float64)
    _append_importance(importance_rows, recent_model, spec["recent"], fold_label, "recent", "v9_control")
    del recent_model, Xtr, Xva, sw_recent, yr, yrlog
    gc.collect()

    Xtr = _matrix(full, spec["occurrence"])
    Xva = _matrix(val, spec["occurrence"])
    clf = CatBoostClassifier(**classifier_params())
    clf.fit(Xtr, pos.astype(np.int8), sample_weight=sw_full, verbose=200)
    p = np.clip(clf.predict_proba(Xva)[:, 1], 0.0, 1.0).astype(np.float64)
    _append_importance(importance_rows, clf, spec["occurrence"], fold_label, "occurrence", "v9_control")
    del clf, Xtr, Xva
    gc.collect()

    Xtr = _matrix(full, spec["positive"], pos)
    Xva = _matrix(val, spec["positive"])
    sev = CatBoostRegressor(**positive_params())
    sev.fit(Xtr, ylog_full[pos], sample_weight=sw_full[pos], verbose=200)
    severity = np.clip(sev.predict(Xva), 0, None).astype(np.float64)
    _append_importance(importance_rows, sev, spec["positive"], fold_label, "positive", "v9_control")
    del sev, Xtr, Xva
    gc.collect()

    final = FIXED_RECENT_WEIGHT * pred_recent + FIXED_HURDLE_WEIGHT * (p * severity)
    final = np.clip(final, 0, None)
    del recent, sw_full
    gc.collect()
    return {"recent": pred_recent, "occurrence": p, "severity": severity, "final": final}

def exact_component_regressor_params(kind: str) -> dict:
    p = positive_params().copy()
    p['random_seed'] = SEED_POSITIVE + (700 if kind == 'frequency_log' else 800)
    return p

def compose_frozen_v13(v9: dict[str, np.ndarray], pred_a: np.ndarray, pred_b: np.ndarray):
    pred_a = np.asarray(pred_a, dtype=np.float64)
    pred_b = np.asarray(pred_b, dtype=np.float64)
    severity = np.clip(pred_a + pred_b, 0.0, None)
    p_occ = np.clip(np.asarray(v9['occurrence'], dtype=np.float64), 0.0, 1.0)
    hurdle = np.clip(p_occ * severity, 0.0, None)
    full_substitution = np.clip(
        FIXED_RECENT_WEIGHT * np.asarray(v9['recent'], dtype=np.float64)
        + FIXED_HURDLE_WEIGHT * hurdle,
        0.0, None,
    )
    base = np.clip(np.asarray(v9['final'], dtype=np.float64), 0.0, None)
    delta = full_substitution - base
    p2_gate = p_occ ** CONTROL_GAMMA
    safe_delta = np.where(delta > 0.0, p2_gate * delta, delta)
    pred = np.clip(base + safe_delta, 0.0, None)
    assert np.isfinite(pred).all() and np.all(pred >= 0)
    return pred, {
        'severity': severity,
        'hurdle': hurdle,
        'full_substitution': full_substitution,
        'delta': delta,
        'p2_gate': p2_gate,
    }

def fit_frequency_head_only(
    full: pl.DataFrame,
    val: pl.DataFrame,
    feature_cols: list[str],
    fold_label: str,
    importance_rows: list[dict],
):
    y = full['target'].to_numpy().astype(np.float64)
    z = np.log1p(np.clip(y, 0, None))
    pos = y > 0
    days = full['future_spend_days'].to_numpy().astype(np.float64)
    a = np.log(days[pos])
    assert np.all(days[pos] >= 1.0)

    Xtr = _matrix(full, feature_cols, pos)
    Xva = _matrix(val, feature_cols)
    sw = recency_weights(full, decay_days=RECENCY_DECAY_ATTR).astype(np.float64)[pos]

    model = CatBoostRegressor(**exact_component_regressor_params('frequency_log'))
    model.fit(Xtr, a, sample_weight=sw, verbose=200)
    pred_a = model.predict(Xva).astype(np.float64)
    _append_importance(
        importance_rows, model, feature_cols, fold_label,
        'log_spend_days', 'V14_FROZEN_A',
    )
    del model, Xtr, Xva
    gc.collect()
    return pred_a

def fit_standard_b_only(
    full: pl.DataFrame,
    val: pl.DataFrame,
    feature_cols: list[str],
    fold_label: str,
    importance_rows: list[dict],
    expert: str,
):
    y = full['target'].to_numpy().astype(np.float64)
    z = np.log1p(np.clip(y, 0, None))
    pos = y > 0
    days = full['future_spend_days'].to_numpy().astype(np.float64)
    b = z[pos] - np.log(days[pos])
    Xtr = _matrix(full, feature_cols, pos)
    Xva = _matrix(val, feature_cols)
    sw = recency_weights(full, decay_days=RECENCY_DECAY_ATTR).astype(np.float64)[pos]

    model = CatBoostRegressor(**exact_component_regressor_params('intensity_log'))
    model.fit(Xtr, b, sample_weight=sw, verbose=200)
    pred_b = model.predict(Xva).astype(np.float64)
    _append_importance(
        importance_rows, model, feature_cols, fold_label,
        'log_intensity_exact', expert,
    )
    del model, Xtr, Xva
    gc.collect()
    return pred_b


In [12]:
def _anchor_date_from_int(day: int) -> str:
    return str(date(1970, 1, 1) + timedelta(days=int(day)))

PURCHASE_AUX_CACHE_VERSION = "v2_next_purchase_nohistory_neutral"

PURCHASE_STATE_FEATURES = [
    'purchase_has_history',
    'purchase_log_count',
    'purchase_last_log_gmv',
    'purchase_mean3_log_gmv',
    'purchase_mean5_log_gmv',
    'purchase_mean10_log_gmv',
    'purchase_std5_log_gmv',
    'purchase_std10_log_gmv',
    'purchase_last_log_ticket',
    'purchase_mean5_log_ticket',
    'purchase_last_gap_days',
    'purchase_mean3_gap_days',
]

PURCHASE_AUX_FEATURE_COLS = [
    'purchase_aux_pred_next_log_gmv',
    'purchase_aux_hist_std5_log_gmv',
    'purchase_aux_has_history',
]

def _rolling_mean_local(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    cs = np.concatenate(([0.0], np.cumsum(x, dtype=np.float64)))
    j = np.arange(n)
    s = np.maximum(0, j - window + 1)
    return ((cs[j + 1] - cs[s]) / (j - s + 1)).astype(np.float32)

def _rolling_std_local(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    cs = np.concatenate(([0.0], np.cumsum(x, dtype=np.float64)))
    cs2 = np.concatenate(([0.0], np.cumsum(x * x, dtype=np.float64)))
    j = np.arange(n)
    s = np.maximum(0, j - window + 1)
    cnt = (j - s + 1).astype(np.float64)
    mean = (cs[j + 1] - cs[s]) / cnt
    mean2 = (cs2[j + 1] - cs2[s]) / cnt
    var = np.maximum(mean2 - mean * mean, 0.0)
    return np.sqrt(var).astype(np.float32)

def _build_purchase_sequence_arrays():
    cache = PURCHASE_AUX_CACHE_DIR / f'purchase_sequence_arrays_{PURCHASE_AUX_CACHE_VERSION}.npz'
    if cache.exists():
        z = np.load(cache)
        return {
            'uid': z['uid'],
            'day': z['day'],
            'state': z['state'],
            'transition_x': z['transition_x'],
            'transition_y': z['transition_y'],
            'transition_label_day': z['transition_label_day'],
            'group_starts': z['group_starts'],
            'group_ends': z['group_ends'],
        }

    p = (
        data.filter(pl.col('gmv') > 0)
        .select([
            'user_id', 'event_date',
            pl.col('gmv').cast(pl.Float64),
            pl.col('to_ord').cast(pl.Float64),
        ])
        .sort(['user_id', 'event_date'])
    )
    assert p.height > 0
    assert p.select(['user_id', 'event_date']).n_unique() == p.height, (
        'Expected one purchase-day row per user/date'
    )

    uid = p['user_id'].to_numpy()
    day = p['event_date'].cast(pl.Int32).to_numpy().astype(np.int32)
    gmv = p['gmv'].to_numpy().astype(np.float64)
    orders = p['to_ord'].to_numpy().astype(np.float64)
    log_gmv = np.log1p(np.clip(gmv, 0, None))
    log_ticket = np.log1p(np.clip(gmv / np.maximum(orders, 1.0), 0, None))

    n = len(uid)
    state = np.zeros((n, len(PURCHASE_STATE_FEATURES)), dtype=np.float32)

    starts = np.r_[0, np.flatnonzero(uid[1:] != uid[:-1]) + 1].astype(np.int64)
    ends = np.r_[starts[1:], n].astype(np.int64)

    for s, e in zip(starts, ends):
        x = log_gmv[s:e]
        t = log_ticket[s:e]
        d = day[s:e].astype(np.float64)
        m = e - s

        state[s:e, 0] = 1.0
        state[s:e, 1] = np.log1p(np.arange(1, m + 1)).astype(np.float32)
        state[s:e, 2] = x.astype(np.float32)
        state[s:e, 3] = _rolling_mean_local(x, 3)
        state[s:e, 4] = _rolling_mean_local(x, 5)
        state[s:e, 5] = _rolling_mean_local(x, 10)
        state[s:e, 6] = _rolling_std_local(x, 5)
        state[s:e, 7] = _rolling_std_local(x, 10)
        state[s:e, 8] = t.astype(np.float32)
        state[s:e, 9] = _rolling_mean_local(t, 5)

        gaps = np.full(m, 999.0, dtype=np.float64)
        if m > 1:
            gaps[1:] = np.diff(d)
        state[s:e, 10] = gaps.astype(np.float32)

        gap_mean = np.full(m, 999.0, dtype=np.float32)
        if m > 1:
            gap_mean[1:] = _rolling_mean_local(gaps[1:], 3)
        state[s:e, 11] = gap_mean

    has_next = uid[:-1] == uid[1:]
    transition_idx = np.flatnonzero(has_next).astype(np.int64)
    label_idx = transition_idx + 1

    transition_x = state[transition_idx].astype(np.float32, copy=False)
    transition_y = log_gmv[label_idx].astype(np.float32, copy=False)
    transition_label_day = day[label_idx].astype(np.int32, copy=False)

    np.savez_compressed(
        cache,
        uid=uid, day=day, state=state,
        transition_x=transition_x,
        transition_y=transition_y,
        transition_label_day=transition_label_day,
        group_starts=starts, group_ends=ends,
    )
    print('Purchase sequence cache:', cache.resolve())
    print('purchase days:', n, 'transitions:', len(transition_y))
    return {
        'uid': uid, 'day': day, 'state': state,
        'transition_x': transition_x, 'transition_y': transition_y,
        'transition_label_day': transition_label_day,
        'group_starts': starts, 'group_ends': ends,
    }

PURCHASE_SEQ = _build_purchase_sequence_arrays()
PURCHASE_AUX_IMPORTANCE_ROWS = []
PURCHASE_AUX_FIT_ROWS = []

def purchase_state_frame_for_anchor(anchor: date) -> pl.DataFrame:
    cp = PURCHASE_AUX_CACHE_DIR / f'purchase_state_{PURCHASE_AUX_CACHE_VERSION}_{anchor}.parquet'
    if cp.exists():
        return pl.read_parquet(cp)

    seq = PURCHASE_SEQ
    anchor_day = (anchor - date(1970, 1, 1)).days
    idxs = []
    for s, e in zip(seq['group_starts'], seq['group_ends']):
        local = np.searchsorted(seq['day'][s:e], anchor_day, side='right') - 1
        if local >= 0:
            idxs.append(int(s + local))
    idxs = np.asarray(idxs, dtype=np.int64)

    state_df = pl.DataFrame({
        'user_id': seq['uid'][idxs],
        **{
            c: seq['state'][idxs, j]
            for j, c in enumerate(PURCHASE_STATE_FEATURES)
        },
    })
    assert state_df['user_id'].n_unique() == state_df.height

    out = (
        all_users.with_row_index('__row_order')
        .join(state_df, on='user_id', how='left', validate='1:1')
        .sort('__row_order').drop('__row_order')
    )

    fill_exprs = []
    for c in PURCHASE_STATE_FEATURES:
        default = 999.0 if c in {'purchase_last_gap_days', 'purchase_mean3_gap_days'} else 0.0
        fill_exprs.append(pl.col(c).fill_null(default).cast(pl.Float32))
    out = out.with_columns(fill_exprs)

    for c in PURCHASE_STATE_FEATURES:
        arr = out[c].to_numpy().astype(np.float64)
        assert np.isfinite(arr).all(), (anchor, c)

    out.write_parquet(cp, compression='zstd', statistics=True)
    return out

def purchase_aux_params():
    return dict(
        iterations=PURCHASE_AUX_ITERATIONS,
        loss_function='RMSE', eval_metric='RMSE',
        depth=PURCHASE_AUX_DEPTH,
        learning_rate=PURCHASE_AUX_LR,
        l2_leaf_reg=10.0,
        random_strength=0.20,
        bootstrap_type='Bayesian',
        bagging_temperature=0.60,
        border_count=BORDER_COUNT,
        random_seed=PURCHASE_AUX_SEED,
        allow_writing_files=False,
        task_type='GPU',
        devices=DEVICES,
        verbose=200,
    )

def purchase_aux_frame_for_anchor(anchor: date) -> pl.DataFrame:
    cp = PURCHASE_AUX_CACHE_DIR / f'purchase_aux_{PURCHASE_AUX_CACHE_VERSION}_{anchor}.parquet'
    if cp.exists():
        out = pl.read_parquet(cp)
        assert out.height == EXPECTED_USERS and out['user_id'].n_unique() == EXPECTED_USERS
        assert all(c in out.columns for c in ['user_id', *PURCHASE_AUX_FEATURE_COLS])
        for c in PURCHASE_AUX_FEATURE_COLS:
            assert out[c].null_count() == 0, (anchor, c)
            assert np.isfinite(out[c].to_numpy()).all(), (anchor, c)
        return out

    seq = PURCHASE_SEQ
    anchor_day = (anchor - date(1970, 1, 1)).days
    m = seq['transition_label_day'] <= anchor_day
    nfit = int(m.sum())
    if nfit < 10_000:
        raise RuntimeError(f'Too few causal purchase transitions for {anchor}: {nfit}')

    Xtr = seq['transition_x'][m].astype(np.float32, copy=False)
    ytr = seq['transition_y'][m].astype(np.float32, copy=False)
    label_day = seq['transition_label_day'][m].astype(np.int32, copy=False)
    assert int(label_day.max()) <= anchor_day, (anchor, int(label_day.max()), anchor_day)
    assert np.isfinite(Xtr).all() and np.isfinite(ytr).all()
    sw = np.exp(-(anchor_day - label_day) / PURCHASE_AUX_DECAY_DAYS).astype(np.float64)
    sw = sw / max(float(sw.mean()), 1e-12)

    state = purchase_state_frame_for_anchor(anchor)
    Xva = state.select(PURCHASE_STATE_FEATURES).to_numpy().astype(np.float32, copy=False)

    model = CatBoostRegressor(**purchase_aux_params())
    model.fit(Xtr, ytr, sample_weight=sw, verbose=200)

    has_history = state['purchase_has_history'].to_numpy().astype(np.float64) > 0.5
    pred = np.zeros(state.height, dtype=np.float64)
    if has_history.any():
        pred[has_history] = model.predict(Xva[has_history]).astype(np.float64)
    assert np.isfinite(pred).all()

    fi = np.asarray(model.get_feature_importance(), dtype=np.float64)
    for rank, j in enumerate(np.argsort(-fi), 1):
        PURCHASE_AUX_IMPORTANCE_ROWS.append({
            'anchor': str(anchor),
            'feature': PURCHASE_STATE_FEATURES[j],
            'importance': float(fi[j]),
            'rank': int(rank),
        })

    PURCHASE_AUX_FIT_ROWS.append({
        'anchor': str(anchor),
        'training_transitions': nfit,
        'training_label_min_day': _anchor_date_from_int(int(label_day.min())),
        'training_label_max_day': _anchor_date_from_int(int(label_day.max())),
        'effective_n': effective_n(sw),
        'pred_mean': float(np.mean(pred)),
        'pred_std': float(np.std(pred)),
        'history_share': float(state['purchase_has_history'].mean()),
    })

    out = pl.DataFrame({
        'user_id': state['user_id'],
        'purchase_aux_pred_next_log_gmv': pred.astype(np.float32),
        'purchase_aux_hist_std5_log_gmv': state['purchase_std5_log_gmv'],
        'purchase_aux_has_history': state['purchase_has_history'],
    })
    assert out.height == EXPECTED_USERS and out['user_id'].n_unique() == EXPECTED_USERS
    for c in PURCHASE_AUX_FEATURE_COLS:
        assert np.isfinite(out[c].to_numpy()).all(), (anchor, c)
    out.write_parquet(cp, compression='zstd', statistics=True)

    del model, Xtr, ytr, Xva, state
    gc.collect()
    return out

def attach_purchase_aux(frame: pl.DataFrame, anchor: date) -> pl.DataFrame:
    aux = purchase_aux_frame_for_anchor(anchor)
    before = frame.height
    out = (
        frame.with_row_index('__row_order')
        .join(aux, on='user_id', how='left', validate='m:1')
        .sort('__row_order').drop('__row_order')
    )
    assert out.height == before
    for c in PURCHASE_AUX_FEATURE_COLS:
        assert out[c].null_count() == 0
        assert np.isfinite(out[c].to_numpy()).all()
    return out

def prediction_percentile_rank(values: np.ndarray) -> np.ndarray:
    v = np.asarray(values, dtype=np.float64)
    assert v.ndim == 1 and len(v) > 0
    assert np.isfinite(v).all()
    sorted_v = np.sort(v)
    rank = np.searchsorted(sorted_v, v, side='right').astype(np.float64) / float(len(v))
    return np.clip(rank, 0.0, 1.0)

def low_value_rank_gate(base_pred_log: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    rank = prediction_percentile_rank(base_pred_log)
    x = (rank - LOW_VALUE_BLOCK_UNTIL_RANK) / (
        LOW_VALUE_FULL_V12_FROM_RANK - LOW_VALUE_BLOCK_UNTIL_RANK
    )
    x = np.clip(x, 0.0, 1.0)
    gate = x * x * (3.0 - 2.0 * x)
    assert np.all((gate >= -1e-12) & (gate <= 1.0 + 1e-12))
    assert np.all(gate[rank <= LOW_VALUE_BLOCK_UNTIL_RANK + 1e-12] <= 1e-12)
    assert np.all(gate[rank >= LOW_VALUE_FULL_V12_FROM_RANK - 1e-12] >= 1.0 - 1e-12)
    return rank, np.clip(gate, 0.0, 1.0)

def compose_with_optional_low(
    v9: dict[str, np.ndarray],
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    apply_low: bool,
):
    parent, diag = compose_frozen_v13(v9, pred_a, pred_b)
    if not apply_low:
        rank, value_gate = low_value_rank_gate(np.asarray(v9['final'], dtype=np.float64))
        return parent, {**diag, 'base_value_rank': rank, 'low_value_gate': value_gate}

    base = np.clip(np.asarray(v9['final'], dtype=np.float64), 0.0, None)
    delta = np.asarray(diag['delta'], dtype=np.float64)
    p2_gate = np.asarray(diag['p2_gate'], dtype=np.float64)
    rank, value_gate = low_value_rank_gate(base)
    low_up_gate = p2_gate * value_gate
    low_delta = np.where(delta > 0.0, low_up_gate * delta, delta)
    low_pred = np.clip(base + low_delta, 0.0, None)

    assert np.all(low_pred <= parent + 1e-10)
    high = rank >= LOW_VALUE_FULL_V12_FROM_RANK - 1e-12
    if high.any():
        assert np.allclose(low_pred[high], parent[high], atol=1e-10, rtol=1e-10)
    bottom = rank <= LOW_VALUE_BLOCK_UNTIL_RANK + 1e-12
    if bottom.any():
        expected = np.clip(base + np.minimum(delta, 0.0), 0.0, None)
        assert np.allclose(low_pred[bottom], expected[bottom], atol=1e-10, rtol=1e-10)

    return low_pred, {
        **diag,
        'base_value_rank': rank,
        'low_value_gate': value_gate,
        'low_up_gate': low_up_gate,
    }


Purchase sequence cache: D:\baboteek_holmotech\artifacts\base_model\_cache\purchase_aux\purchase_sequence_arrays_v2_next_purchase_nohistory_neutral.npz
purchase days: 4736907 transitions: 4517739


## OOF

In [15]:
importance_rows = []
oof_parts = []

B3_FEATURE_COLS = BASE_VALUE_FEATURE_COLS + PURCHASE_AUX_FEATURE_COLS
assert len(B3_FEATURE_COLS) == len(set(B3_FEATURE_COLS))

eval_plan = [
    ("primary", PRIMARY_OUTERS),
    ("stress", STRESS_OUTERS),
]

for split, outers in eval_plan:
    for outer in outers:
        print(f"{split} | {outer}")

        eligible = _eligible_train_anchors_strict(outer)
        train_parts = []

        for anchor in eligible:
            frame = build_v12_frame(anchor, with_target=True)
            frame = attach_purchase_aux(frame, anchor)
            train_parts.append(frame)

        full = pl.concat(train_parts, how="vertical_relaxed", rechunk=False)
        del train_parts
        gc.collect()

        val = build_v12_frame(outer, with_target=True)
        val = attach_purchase_aux(val, outer)

        v9 = fit_predict_v9_control(
            full,
            val,
            eligible,
            f"{outer}_base",
            importance_rows,
        )
        pred_a = fit_frequency_head_only(
            full,
            val,
            BASE_VALUE_FEATURE_COLS,
            str(outer),
            importance_rows,
        )
        pred_b = fit_standard_b_only(
            full,
            val,
            B3_FEATURE_COLS,
            str(outer),
            importance_rows,
            CANDIDATE,
        )
        pred_log, _ = compose_with_optional_low(v9, pred_a, pred_b, True)

        assert len(pred_log) == EXPECTED_USERS
        assert np.isfinite(pred_log).all()
        assert np.all(pred_log >= 0)

        oof_parts.append(
            pl.DataFrame({
                "user_id": val["user_id"],
                "split": np.full(EXPECTED_USERS, split),
                "fold": np.full(EXPECTED_USERS, str(outer)),
                "target": val["target"],
                "future_spend_days": val["future_spend_days"],
                "v9_occurrence": v9["occurrence"],
                "v9_recent_log": v9["recent"],
                "v9_base_log": v9["final"],
                "pred_a_frozen": pred_a,
                "pred_b_b3": pred_b,
                f"pred_log__{CANDIDATE}": pred_log,
                "purchase_aux_pred_next_log_gmv": val["purchase_aux_pred_next_log_gmv"],
                "purchase_aux_hist_std5_log_gmv": val["purchase_aux_hist_std5_log_gmv"],
                "purchase_aux_has_history": val["purchase_aux_has_history"],
            })
        )

        del full, val, v9, pred_a, pred_b, pred_log
        gc.collect()

OOF = pl.concat(oof_parts, how="vertical_relaxed")
OOF_PATH = ARTIFACT_ROOT / "oof_candidate_predictions.parquet"
OOF.write_parquet(OOF_PATH, compression="zstd")

if importance_rows:
    pl.DataFrame(importance_rows).write_csv(ARTIFACT_ROOT / "feature_importance.csv")

print("OOF:", OOF_PATH.resolve(), OOF.shape)


primary | 2025-07-30
0:	learn: 1.0485535	total: 18ms	remaining: 8.96s
200:	learn: 1.0075795	total: 766ms	remaining: 1.14s
400:	learn: 1.0055295	total: 1.52s	remaining: 375ms
499:	learn: 1.0046082	total: 1.9s	remaining: 0us
0:	learn: 1.0613942	total: 5.62ms	remaining: 2.81s
200:	learn: 1.0161913	total: 871ms	remaining: 1.29s
400:	learn: 1.0149675	total: 1.75s	remaining: 433ms
499:	learn: 1.0144633	total: 2.23s	remaining: 0us
0:	learn: 1.0670274	total: 5.06ms	remaining: 2.52s
200:	learn: 1.0192234	total: 947ms	remaining: 1.41s
400:	learn: 1.0183631	total: 1.91s	remaining: 472ms
499:	learn: 1.0179955	total: 2.4s	remaining: 0us
0:	learn: 1.0680095	total: 6.92ms	remaining: 3.46s
200:	learn: 1.0181384	total: 1.18s	remaining: 1.76s
400:	learn: 1.0173961	total: 2.3s	remaining: 567ms
499:	learn: 1.0170991	total: 2.85s	remaining: 0us
0:	learn: 1.0674696	total: 7.93ms	remaining: 3.96s
200:	learn: 1.0169437	total: 1.38s	remaining: 2.05s
400:	learn: 1.0163293	total: 2.77s	remaining: 685ms
499:	lear

## Финальный fit

In [17]:
full_train_anchors = [
    anchor
    for anchor in V9_CANONICAL_POOL
    if anchor + timedelta(days=HORIZON) <= max_date
]
assert full_train_anchors == V9_CANONICAL_POOL

parts = []
for i, anchor in enumerate(full_train_anchors, 1):
    print(f"final train [{i}/{len(full_train_anchors)}] | {anchor}")
    frame = build_v12_frame(anchor, with_target=True)
    frame = attach_purchase_aux(frame, anchor)
    parts.append(frame)

full = pl.concat(parts, how="vertical_relaxed", rechunk=False)
del parts
gc.collect()

final = build_v12_frame(max_date, with_target=False)
final = attach_purchase_aux(final, max_date)

v9 = fit_predict_v9_control(
    full,
    final,
    full_train_anchors,
    "final",
    importance_rows,
)
pred_a = fit_frequency_head_only(
    full,
    final,
    BASE_VALUE_FEATURE_COLS,
    "final",
    importance_rows,
)
pred_b = fit_standard_b_only(
    full,
    final,
    B3_FEATURE_COLS,
    "final",
    importance_rows,
    CANDIDATE,
)
pred_log, _ = compose_with_optional_low(v9, pred_a, pred_b, True)

final_out = pl.DataFrame({
    "user_id": final["user_id"],
    "v9_occurrence": v9["occurrence"],
    "v9_recent_log": v9["recent"],
    "v9_base_log": v9["final"],
    "pred_a_frozen": pred_a,
    "pred_b_b3": pred_b,
    f"pred_log__{CANDIDATE}": pred_log,
    "purchase_aux_pred_next_log_gmv": final["purchase_aux_pred_next_log_gmv"],
    "purchase_aux_hist_std5_log_gmv": final["purchase_aux_hist_std5_log_gmv"],
    "purchase_aux_has_history": final["purchase_aux_has_history"],
})

FINAL_PATH = ARTIFACT_ROOT / "final_candidate_predictions.parquet"
final_out.write_parquet(FINAL_PATH, compression="zstd")

pred_gmv = np.expm1(np.clip(pred_log, 0.0, 30.0))
pred_gmv = np.nan_to_num(pred_gmv, nan=0.0, posinf=0.0, neginf=0.0)
pred_gmv = np.clip(pred_gmv, 0.0, None)

submission_pred = pl.DataFrame({
    "user_id": final["user_id"],
    "predict": pred_gmv,
})
submission = (
    submit_template.select("user_id")
    .with_row_index("__order")
    .join(submission_pred, on="user_id", how="left", validate="1:1")
    .sort("__order")
    .drop("__order")
    .with_columns(
        pl.col("predict")
        .fill_nan(0.0)
        .fill_null(0.0)
        .clip(lower_bound=0.0)
        .cast(pl.Float64)
    )
)

assert final_out.height == EXPECTED_USERS
assert submission.height == EXPECTED_USERS
assert submission.columns == ["user_id", "predict"]
assert submission["predict"].null_count() == 0
assert np.isfinite(submission["predict"].to_numpy()).all()

print("Final:", FINAL_PATH.resolve())
print("Submission:", SUBMISSION_PATH.resolve())

del full, final, v9, pred_a, pred_b, pred_log
gc.collect()


final train [1/11] | 2025-02-13
final train [2/11] | 2025-03-26
final train [3/11] | 2025-05-07
final train [4/11] | 2025-06-18
final train [5/11] | 2025-07-30
final train [6/11] | 2025-09-10
final train [7/11] | 2025-10-22
final train [8/11] | 2025-11-12
final train [9/11] | 2025-12-03
final train [10/11] | 2025-12-15
final train [11/11] | 2026-01-14
0:	learn: 2.3051197	total: 98.5ms	remaining: 1m 52s
200:	learn: 1.7162218	total: 4.91s	remaining: 22.9s
400:	learn: 1.7106009	total: 9.47s	remaining: 17.4s
600:	learn: 1.7064624	total: 13.9s	remaining: 12.4s
800:	learn: 1.7029565	total: 18.3s	remaining: 7.71s
1000:	learn: 1.6995616	total: 22.8s	remaining: 3.12s
1137:	learn: 1.6973780	total: 25.8s	remaining: 0us
0:	learn: 0.6728969	total: 58.9ms	remaining: 48.3s
200:	learn: 0.4628604	total: 10.9s	remaining: 33.6s
400:	learn: 0.4612293	total: 21.5s	remaining: 22.5s
600:	learn: 0.4600925	total: 31.7s	remaining: 11.6s
800:	learn: 0.4590850	total: 41.9s	remaining: 993ms
819:	learn: 0.4589948	t

0